# 지정 리그 + 포지션 리드 + 팀 리드 전환율

사용자 가설: 인프라가 잘 갖춰진 일부 지역으로 학습 범위를 좁히고, 포지션의 리드와 팀의 리드 획득/전환 능력을 더 잘 활용하면 예측이 개선될 수 있다.

확인한 의미: 피처의 영향력을 작게 제한하자는 뜻이 아니라 **더 잘 활용하자**는 뜻. 피처 값을 임의로 크게 곱하지 않고, 새로운 정보를 분리해 표준화 후 학습한다.

## 데이터 범위

사용자가 열거한 5지역을 CSV의 LCK, LEC, LTA N, LPL, VCS로 대응시켰다. 베트남은 사용자 확인에 따라 **VCS만**, LCP는 제외한다. 북미는 LTA N만, 다른 지역도 섞인 LTA 통합 대회는 이번에 포함하지 않는다.

| 리그 | 사용 세트 | 제외 |
|---|---:|---|
| LCK | 555 | 0 |
| LEC | 306 | 0 |
| LTA N | 214 | 0 |
| LPL | 0 | 805: partial, 15분 골드/XP/CS 통계 결측 |
| VCS | 137 | 4: 선수 ID 결측 |

총 **1,212세트**, 초기 fitting 1,002세트. 2025년 마지막 LCK 33시리즈에서만 epoch를 고르고 2025년 전체로 refit한다. 2026년 동일 LCK 186시리즈에서 frozen 모델을 평가한다. 5개 지역을 요청했으나 실제 적격 학습은 4개 리그이며, LPL을 0값으로 만들어 포함하지 않는다.

## 피처 정의

### 기존 선수 13개

과거 5세트 평균 승률, 15분 골드/XP/CS 차이, 분당 킬/데스/어시스트, DPM, 피해/골드 점유율, CSPM, 분당 시야, 기록량. 기존 LCK 입력과 동일함을 검사했다.

### 추가 선수 3개 → 16개

- 최근 5세트 중 **15분 골드 차이 > 0**인 비율.
- 최근 5세트 중 **15분 XP 차이 > 0**인 비율.
- 최근 5세트 중 **15분 CS 차이 > 0**인 비율.

선수 ID별 shift(1) 뒤 rolling5를 계산한다. 골드/XP/CS 평균과 리드 빈도는 다른 정보다. 빈도는 리드의 크기를 표현하지 않으며 기존 평균을 대체하지 않고 추가한다.

### 팀 리드 4개

최근 5세트 팀 기준 평균 15분 골드 차이, 양수 골드 리드율, 1,000골드 이상 리드율, 기록량/5. 선수 branch와 분리된 선형 팀 점수 branch로 넣고 둘을 합쳐 상대 팀 점수와 비교한다.

### 팀 승리 전환율 1개

`(과거 5세트 중 15분 리드하고 승리한 횟수 + 1) / (과거 5세트 중 15분 리드한 횟수 + 2)`.

여기서 리드는 팀 golddiffat15 > 0. 1번 리드해서 1번 이긴 경우 100%라고 확신하지 않도록 Beta(1,1) smoothing을 사용한다. 리드 경험이 없으면 0.5. **현재 경기의 리드나 승패는 들어가지 않는다.** 이를 실제 경기 결과를 바꾼 합성 데이터 검사로 확인했다.

## 비교 조건

A 포지션별 합산 구조를 기준으로 단계적으로 추가한다. 선수 인코더의 입력 차원과 팀 branch 추가로 파라미터는 213→237→241→242개다. 완전히 같은 파라미터 수의 인과적 비교는 아니다. seed42/43/44, Adam.001, max500/patience30, 같은 2025 LCK validation 시리즈 log loss로 epoch 선택.

LCK 선수 이력은 LCK-only로 유지한다. non-LCK 선수는 선택한 non-LCK 리그 안에서 과거 기록을 연결하고, non-LCK 팀 기록은 league+teamname 기준이다. scaler/결측 중앙값은 각 fitting 데이터에서만 계산한다.

세 seed 세트 확률 평균 → iid Bo3/Bo5 변환. 첫 세트 명단이 알려진 시점을 가정한다. 2026년은 여러 번 열람한 회고 평가 자료이며 독립 test는 아니다.


In [1]:
from pathlib import Path
import sys,json
import pandas as pd
from IPython.display import display
ROOT=Path('/Users/seungyunmok/Developer/LOL_ML')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
RESULTS=ROOT/'notebook/experiments/12_selected_league_leads'

## 재실행

False는 저장된 결과를 읽고, True는 전체 실험을 다시 실행한다.

In [2]:
RUN_EXPERIMENT=False
if RUN_EXPERIMENT:
    from backend.training.selected_league_leads import run
    run(['LCK','LEC','LTA N','LPL','VCS'])

In [3]:
display(pd.read_csv(RESULTS/'league_audit.csv'))
print(json.dumps(json.loads((RESULTS/'lock.json').read_text()),indent=2))

,league,eligible,missing_player_id,not_complete
0,LCK,555,0,0
1,LEC,306,0,0
2,LPL,0,0,805
3,LTA N,214,0,0
4,VCS,137,4,0


{
  "train_sets": 1212,
  "fit_sets": 1002,
  "validation_series": 33,
  "leagues": {
    "LCK": 555,
    "LEC": 306,
    "LTA N": 214,
    "VCS": 137
  },
  "variants": {
    "selected_baseline": {
      "epochs": {
        "42": 1,
        "43": 7,
        "44": 74
      },
      "player_dim": 13,
      "team_dim": 0,
      "parameters": 213
    },
    "player_leads": {
      "epochs": {
        "42": 1,
        "43": 1,
        "44": 33
      },
      "player_dim": 16,
      "team_dim": 0,
      "parameters": 237
    },
    "team_lead_frequency": {
      "epochs": {
        "42": 100,
        "43": 1,
        "44": 35
      },
      "player_dim": 16,
      "team_dim": 4,
      "parameters": 241
    },
    "team_lead_conversion": {
      "epochs": {
        "42": 95,
        "43": 1,
        "44": 40
      },
      "player_dim": 16,
      "team_dim": 5,
      "parameters": 242
    }
  }
}


## 2026년 동일한 186시리즈 결과

In [4]:
scores=pd.read_csv(RESULTS/'evaluation_2026.csv')
display(scores.round(4))

,variant,n,correct,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,selected_baseline,186,114,0.6129,0.6397,0.6784,0.2427,0.4176,0.5916
1,player_leads,186,111,0.5968,0.6282,0.6846,0.2457,0.4337,0.5546
2,team_lead_frequency,186,95,0.5108,0.4966,0.7183,0.2617,0.2203,0.7296
3,team_lead_conversion,186,102,0.5484,0.5587,0.7021,0.2533,0.1478,0.8017


## 판단

| 구성 | 정답/186 | 정확도 | Log loss |
|---|---:|---:|---:|
| 이전 LCK-only A (참조) | 118 | 63.44% | 0.6754 |
| 지정 리그 + 기존 선수13 | 114 | 61.29% | 0.6784 |
| + 포지션 리드 빈도3 | 111 | 59.68% | 0.6846 |
| + 팀 리드지표4 | 95 | 51.08% | 0.7183 |
| + 팀 승리 전환율1 | 102 | 54.84% | 0.7021 |

현재 구현은 가설을 지지하지 않았다. 전환율은 팀 리드 단계보다 7개 더 맞혔지만, 기준 모델들에는 못 미쳤다. 따라서 현재 기준은 LCK-only A 유지.

짧은 5세트 전환율의 표본 변동, 선수 리드 정보와 팀 정보의 중복, 리그 분포 차이, 작은 validation33은 가능한 설명이다. 이번 비교만으로 어느 것이 원인인지 확정하지 않는다. 팀 피처가 본질적으로 무용하거나 선수 리드가 중요하지 않다는 결론도 아니다.

다음 실험 후보는 팀 전환율의 관측 기간을 늘리고, 다른 변경 없이 그 효과를 분리하는 것이다. 아직 실행하지 않았다. 가중치를 임의로 크게 주거나 2026년 점수를 보고 threshold를 바꾸지 않았다. 배포 변경 없음.


In [5]:
display(pd.read_csv(RESULTS/'monthly_2026.csv').round(4))
print(json.dumps(json.loads((RESULTS/'manifest.json').read_text()),indent=2))

,variant,month,n,accuracy,roc_auc,log_loss,brier_score,probability_min,probability_max
0,player_leads,2026-01,24,0.7083,0.8815,0.6715,0.2392,0.4642,0.5473
1,player_leads,2026-02,15,0.6000,0.5455,0.6934,0.2501,0.4604,0.5343
2,player_leads,2026-03,1,1.0000,NaN,0.5686,0.1881,0.4337,0.4337
3,player_leads,2026-04,44,0.6364,0.6474,0.6841,0.2455,0.4648,0.5404
4,player_leads,2026-05,46,0.5870,0.6788,0.6811,0.2440,0.4576,0.5436
5,player_leads,2026-06,5,0.4000,0.3333,0.6954,0.2511,0.4786,0.5523
6,player_leads,2026-07,6,0.6667,0.5556,0.6969,0.2519,0.4833,0.5235
7,player_leads,2026-08,38,0.5263,0.5623,0.6914,0.2491,0.4418,0.5450
8,player_leads,2026-09,7,0.4286,0.3000,0.6989,0.2529,0.4711,0.5546
9,selected_baseline,2026-01,24,0.6667,0.7926,0.6597,0.2333,0.4452,0.5696


{
  "checkpoint_sha256": "605d31ccc7bfd120592a43ca0073f6e4f41321f05422901a7abcf0fcfd15892d",
  "source_sha256": {
    "2025": "c9a158b9e0a965a47d31d3674c127a26f75e6c91a324bd1858e4784b1336214a",
    "2026": "024330eb7a03e07c1aba55e17abf2e55f8e79e46e42e3f47389194173e59730a"
  },
  "checks": [
    "original LCK13 inputs identical",
    "appending2026 leaves2025 features unchanged",
    "team swap symmetry",
    "checkpoint frozen"
  ]
}
